In [ ]:
#THIS NOTEBOOK IS THE BASELINE PREDICTION MODEL FOR FUSION ENTHALPIES

#import sys
#sys.path.append("..")
from chemprop_utils import load_and_clean_data, build_splits, build_and_train_mpnn, predict_test_set    #IMPORTS REQUIRED FUNCTIONS FROM THE FUNCTION SCRIPT
from pathlib import Path                    #IMPORTS THE FILE PATH HANDLING LIBRARY
import pandas as pd                         #IMPORTS THE DATA FRAME HANDLING LIBRARY
from lightning import pytorch as pl         #IMPORTS ADDITIONAL LIBARIES USED BY FUNCTIONS

pl.seed_everything(0, workers=True)         #INITILIAZES THE SEED AND NUMBER OF WORKERS TO 0

# LOCKED BASELINE CONFIGURATION BASED ON SENSITIVITY ANALYSIS

# SPLIT: SCAFFOLD_BALANCED SPLIT WILL BE USED AS IT OUTPERFORMED RANDOM SPLIT RUNS (RMSE ~2.35 VS RMSE ~4)
# RMSE WILL BE USED AS TRAINING OBJECTIVE. SENSITIVITY RSEULTS WERE INCONCLUSIVE WITH REGARDS TO AN OUTRIGHT OPTIMAL TRAINING OBJECTIVE (MINOR DIFFERENCES ACROSS RUNS), BUT RMSE IS CONSIDERED STANDARD
# DROPOUT OF 0.4 WILL BE USED AS IT RESULTED IN THE LOWEST MAE OF ALL TESTED VALUES
# NO FOLDING WILL BE DONE (SINGLE 80/10/10 SPLIT) TO INCREASE RUN SPEED

input_path = Path("RAW DATA") / "dHfus_DB.csv"          #IMPORTS THE RAW DATA FROM SPECIFIED FILE PATH
smiles_column = "smiles"                                #DEFINES THE SMILES COLUMN
target_columns = ["dHfus [kcal/mol]"]                   #DEFINES THE FUSION ENTHALPY COLUMN AS THE TARGET

df_clean, all_data, mols = load_and_clean_data(input_path, smiles_column, target_columns)       #RUNS THE DATA CLEANING AND LOADING FUNCTION
splits = build_splits(all_data, mols, split_types=("scaffold_balanced",))                       #CALLS THE SPLITTING FUNCTION AND USES SCAFFOLD SPLITTING 

result_baseline = build_and_train_mpnn(                 #CALLS THE MPNN BUILDING FUNCTION AND SPECIFIES THAT SCAFFOLD SPLITTING IS USED FOR TRAINING, VALIDATION AND TEST SPLITTING
    splits["scaffold_balanced"]["train"],
    splits["scaffold_balanced"]["val"],
    splits["scaffold_balanced"]["test"],
    dropout=0.4,
    run_name="dHfus_OFFICIAL_BASELINE",
)

test_metrics = result_baseline["test_results"][0]        #EXTRACTS THE ERROR METRICS OF THE TEST RESULTS
print(f"OFFICIAL BASELINE — RMSE: {test_metrics['test/rmse']:.3f}, MAE: {test_metrics['test/mae']:.3f}")

baseline_predictions = predict_test_set(result_baseline, run_name="dHfus_OFFICIAL_BASELINE")    #CALLS THE PREDICTION FUNCTION
baseline_predictions.to_csv("dHfus_baseline_predictions.csv", index=False)                      #SAVES THE BASELINE PREDICTIONS

#NOTE: RESULTS AND SPLITS MUST BE SAVED TO BE USED IN SUBSEQUENT NOTEBOOKS
import pickle                                           #IMPORT LIBRARY FOR DATA SAVING
with open("dHfus_baseline_splits.pkl", "wb") as f:      #OPENS A NEW FILE FOR WRITING 
    pickle.dump(splits, f)                              #SAVE THE SPLITS AND RESULTS BY WRITING TO THE FILE

print("Baseline locked in. Predictions and splits saved for downstream sub-question notebooks.")

Seed set to 0


Loaded 5184 rows, dropped 527 with missing target, 4657 remain


[16:00:50] WARNING: not removing hydrogen atom without neighbors
[16:00:50] WARNING: not removing hydrogen atom without neighbors
[16:00:50] WARNING: not removing hydrogen atom without neighbors
[16:00:50] WARNING: not removing hydrogen atom without neighbors
[16:00:50] WARNING: not removing hydrogen atom without neighbors
[16:00:50] WARNING: not removing hydrogen atom without neighbors
[16:00:50] WARNING: not removing hydrogen atom without neighbors
[16:00:50] Unusual charge on atom 6 number of radical electrons set to zero
The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
c:\Users\Jaco Olivier\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\astartes\samplers\extrapolation\scaffold.py:48: NoMatchingScaffold: No matching scaffold was found for the 1308 molecules corresponding to indices {3, 6, 9, 12, 13, 22, 29, 39, 41, 50, 56, 62, 63, 70, 71, 83, 85, 86, 88, 90, 95, 97, 99, 106, 109, 118, 121, 129, 132, 133, 135, 138, 148, 149, 150, 152

scaffold_balanced: train=3727, val=465, test=465


c:\Users\Jaco Olivier\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\Jaco Olivier\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

... (7 outputs left out)

Output()

OFFICIAL BASELINE — RMSE: 2.138, MAE: 1.532


c:\Users\Jaco Olivier\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Baseline locked in. Predictions and splits saved for downstream sub-question notebooks.
